# Qwen3-0.6B project: unified presentation demo

This notebook walks through **training → generalization → inference → tiny Transformer** in one place. It reads the saved results and verifies the published LoRA checkpoint. It does **not** train a model or rerun the 900-question inference benchmarks. The tiny Transformer section will show results automatically after the team adds them to `tiny_transformer/results/`.

For a short live Base-versus-LoRA generation, open the separate [generalization demo](https://colab.research.google.com/github/oscarwyx/LLM-generalization/blob/main/generalization/notebooks/generalization_presentation_demo.ipynb) on a GPU. This overview notebook runs on a CPU Colab runtime.

In [ ]:
# 1. Fetch the current combined repository and verify the published checkpoint.
from pathlib import Path
import subprocess, sys, json
ROOT = Path('/content/LLM-generalization')
if not (ROOT / '.git').is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/oscarwyx/LLM-generalization.git', str(ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(ROOT), 'checkout', '--detach', 'origin/main'], check=True)
COMMIT = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip()
TRAINING = ROOT / 'training'
sys.path.insert(0, str(TRAINING / 'src'))
from genqa.pretrained import verify_published
verification = verify_published(TRAINING)
assert verification['status'] == 'passed' and verification['counts']['test'] == 900
print('Repository commit:', COMMIT)
print('Checkpoint and fixed test set verified:', verification['verified_files'], 'files')

## Data and evaluation setup — presenter script

“We used the existing SQuAD 1.1 question-answering data through the MRQA release. We did not collect new data. The published training run used 2,048 training questions, 256 validation questions, and a fixed evaluation set of 900 questions from 45 articles. We kept evaluation articles separate from training. All three completed project parts share the same published model setup and evaluation questions, making their results comparable. Our generalization study first used 90 questions, then validated on the remaining 810.”

This is approximately 30 seconds of narration. The 900-question set comes from the public SQuAD development data, not a hidden test set.

## Part 1 — Training

The team evaluated the same fixed 900-question SQuAD 1.1 test set using a base model, three-shot prompting, and a seed-17 epoch-2 LoRA adapter. The checkpoint is the 20 MB adapter in `training/published/seed17_epoch2/adapter/`; the pinned Qwen3-0.6B base model is downloaded only for live GPU inference.

In [ ]:
# 2. Show the recorded training outcome.
import pandas as pd
from IPython.display import display, Image, Markdown
training_metrics = pd.read_csv(TRAINING / 'published/seed17_epoch2/reference/metrics.csv')
test_metrics = training_metrics[training_metrics.phase.eq('test')][['method', 'n', 'EM (%)', 'F1 (%)']]
assert len(test_metrics) == 3 and set(test_metrics.method) == {'Baseline', 'Prompting', 'LoRA'}
display(test_metrics.round(2))
display(Image(filename=str(TRAINING / 'published/seed17_epoch2/reference/results.png'), width=820))

## Part 2 — Generalization

We tested whether irrelevant text **before** the answer-bearing passage degrades performance. The initial 90-question study and the remaining 810-question validation are reported separately. The original 90-question prediction of an eight-point LoRA EM decline was not met; its exploratory model-difference interval crossed zero.

In [ ]:
# 3. Show paired generalization results and both figures.
G = ROOT / 'generalization'
initial = pd.read_csv(G / 'repro/data/initial_90/exploratory_interaction.csv')
validation = pd.read_csv(G / 'repro/data/validation_810/paired_effects.csv')
print('Initial 90: exploratory LoRA-minus-Baseline EM change (percentage points)')
display(initial[initial.metric.eq('em')].round(2))
print('Remaining 810: paired EM changes (percentage points)')
em = validation[validation.metric.eq('em')]
display(em.round(2))
primary = em[em.contrast.eq('LoRA_minus_Baseline')].iloc[0]
assert round(primary.change_pp, 2) == 3.40
assert primary.ci95_low_pp > 0
for title, file in [
    ('Initial 90: length effects', G / 'presentation/overleaf/figures/initial_90_length_effects.png'),
    ('Validation 810: paired effects', G / 'presentation/overleaf/figures/validation_810_quantitative.png')]:
    display(Markdown('### ' + title))
    display(Image(filename=str(file), width=820))
display(Markdown('**Claim:** this one LoRA checkpoint lost fewer EM points than the base model in the 810-question SQuAD prepended-context validation. These are new questions from the same 45 articles, not a new domain.'))

## Part 3 — Inference efficiency

The teammate benchmarked KV caching, batch size, and decoding strategy on the same published 900-question test set. These are saved measurements from an NVIDIA T4 in FP16. The figures and CSVs are shown here; the full benchmark is not rerun during the talk.

In [ ]:
# 4. Show the teammate's measured efficiency and quality tradeoffs.
import json
I = ROOT / 'inference'
source_manifest = verification['manifest_sha256']
for folder in ['results-KV cache', 'results-batch size', 'results-decoding strategy']:
    config = json.loads((I / folder / 'run_config.json').read_text())
    assert config['publication_manifest_sha256'] == source_manifest, folder

cache = pd.read_csv(I / 'results-KV cache/summary.csv')
batch = pd.read_csv(I / 'results-batch size/summary.csv')
decoding = pd.read_csv(I / 'results-decoding strategy/summary.csv')
print('KV cache: generation throughput and memory')
display(cache[['kv_cache', 'questions_per_s', 'peak_extra_allocated_mib', 'em_pct']].round(2))
display(Image(filename=str(I / 'results-KV cache/benchmark.png'), width=820))
print('Batch size: throughput versus peak memory')
display(batch[['batch_size', 'questions_per_s', 'peak_allocated_mib', 'em_pct']].round(2))
display(Image(filename=str(I / 'results-batch size/batch_size_benchmark.png'), width=820))
print('Decoding: speed and answer quality')
display(decoding[['num_beams', 'questions_per_s', 'em_pct', 'f1_pct']].round(2))
display(Image(filename=str(I / 'results-decoding strategy/decoding_strategy_benchmark.png'), width=820))

## Part 4 — Tiny Transformer

This component has not been delivered yet. The cell below shows a clear **pending** status today. When the team adds `tiny_transformer/results/summary.csv` and, optionally, `tiny_transformer/results/figure.png`, the same notebook will display them on the next run. Add the implementation, configuration, and reproduction steps to that folder's README.

In [ ]:
# 5. Automatically pick up the future tiny Transformer results.
T = ROOT / 'tiny_transformer'
summary_path = T / 'results/summary.csv'
figure_path = T / 'results/figure.png'
if summary_path.is_file():
    print('Tiny Transformer results:')
    display(pd.read_csv(summary_path))
    if figure_path.is_file():
        display(Image(filename=str(figure_path), width=820))
else:
    display(Markdown('**Tiny Transformer: pending.** No training result or checkpoint has been added to the repository yet.'))

## Presentation takeaway

- **Training:** the published epoch-2 LoRA checkpoint reached 77.11% EM on the fixed test set.
- **Generalization:** in the 810-question validation, its EM decline under prepended irrelevant context was 3.40 points smaller than the base model's, with a 95% article-cluster interval from +0.93 to +5.80 points.
- **Inference:** KV caching improved throughput at a memory cost; larger batches improved throughput; greedy decoding gave the strongest QA scores among the tested strategies.
- **Tiny Transformer:** add the team's result after it is completed. Do not present the placeholder as an experiment.